In the following, we provide a code for the 4th step of the proof with genus g=2.

In [1]:
import numpy as np
import pandas as pd
from scipy.linalg import companion
from numpy.linalg import matrix_power
from tqdm import tqdm
from sympy import divisors

We first compute the list of polynomials Q as in step 6.

In [2]:
degree=6 #degree of the polynomials Q is 2g=6
coeff_candidate=[1,0,0,-1,0,-1,0,0,-1] #candidate polynomial is x^8-x^5-x^3-1
lmin=max(abs(np.roots(coeff_candidate)))

We now compute the bounds on the power sums. Since the polynomials we are looking for are skew-reciprocal, we only need to compute half the coefficients and hence half the power sums.

In [3]:
p_up=np.array([],dtype=int)
for i in range(1,int(np.floor(degree/2))+1):
    p_up=np.append(p_up,[int(np.floor(degree/2*(lmin**i+lmin**(-i))))])

print(p_up) #upper bounds on the power sums

[6 6 7]


In [4]:
p_low=np.array([],dtype=int)
for i in range(1,int(np.floor(degree/2))+1):
    p_low=np.append(p_low,[int(np.ceil(min(2-degree,-(degree/2-2)*lmin**i-degree/2*lmin**(-i))))])

print(p_low)

[-4 -4 -4]


We now use the Newton's formula to compute the coefficients and check if the largest root is smaller lmin in absolute value.

In [5]:
working=[]
for p1 in tqdm(range(p_up[0],p_low[0]-1,-1)):
    c1=-p1
    p2_first=(-c1*p1)%2
    for p2 in range(p_up[1]-(p_up[1]-p2_first)%2,p_low[1]-1,-2):
        c2=(p1**2-p2)//2
        p3_first=(-c1*p2-c2*p1)%3
        for p3 in range(p_up[2]-(p_up[2]-p3_first)%3,p_low[2]-1,-3):
            c3=(-c1*p2-c2*p1-p3)//3
            coeff=[1,c1,c2,c3,-c2,c1,-1] #higher power first
            roots=np.roots(coeff)
            l=max(abs(roots))
            if l<lmin+0.00001 and np.polyval(coeff,lmin)>-0.00001:
                c4=-c2
                c5=c1
                coeff.extend([l,p1,p2,p3])
                working.append(coeff) 
polyQ=pd.DataFrame(working,columns=['c0','c1','c2','c3','c4','c5','c6','max root','p1','p2','p3'])

100%|██████████| 11/11 [00:00<00:00, 261.13it/s]


We also compute a few power sums.

In [6]:
maxpowersum=24

zeros=np.zeros(len(polyQ.index),dtype=np.int64)
for i in range(4,maxpowersum):
    polyQ.insert(len(polyQ.columns),f'p{i}',zeros)

for idq in polyQ.index:
    c=[polyQ[f'c{i}'][idq] for i in range(0,7,1)]
    C=companion(c)
    B=companion(c)
    polyQ.at[idq,'p1']=np.matrix.trace(C)
    for i in range(2,maxpowersum):
        B=np.matmul(B,C)
        polyQ.at[idq,f'p{i}']=np.matrix.trace(B)

Using the same algorithme, we compute the list of polynomials P.

In [7]:
degree=12 #6g-6=12

p_up=np.array([],dtype=int)
for i in range(1,int(np.floor(degree/2))+1):
    p_up=np.append(p_up,[int(np.floor(degree/2*(lmin**i+lmin**(-i))))])

print(p_up) #upper bounds on the power sums

[12 13 14 17 20 24]


In [8]:
p_low=np.array([],dtype=int)
for i in range(1,int(np.floor(degree/2))+1):
    p_low=np.append(p_low,[int(np.ceil(min(2-degree,-(degree/2-2)*lmin**i-degree/2*lmin**(-i))))])

print(p_low)

[-10 -10 -10 -12 -14 -16]


By Lemma 7, we know the traces of the odd power of the homeomorphisms must be non-negativ. Hence, knowing the power sums for Q, we can redifine the lower bounds for P.

In [9]:
p_low[0]=0
p_low[2]=-3
p_low[4]=-5

We now use the Newton's formula to compute the coefficients and check if the largest root is smaller lmin in absolute value and that it is Perron.

In [10]:
working=[]
for p1 in tqdm(range(p_up[0],p_low[0]-1,-1)):
    c1=-p1
    p2_first=(-c1*p1)%2
    for p2 in range(p_up[1]-(p_up[1]-p2_first)%2,p_low[1]-1,-2):
        c2=(p1**2-p2)//2
        p3_first=(-c1*p2-c2*p1)%3
        for p3 in range(p_up[2]-(p_up[2]-p3_first)%3,p_low[2]-1,-3):
            c3=(-c1*p2-c2*p1-p3)//3
            p4_first=(-c1*p3-c2*p2-c3*p1)%4
            for p4 in range(p_up[3]-(p_up[3]-p4_first)%4,p_low[3]-1,-4):
                c4=(-c1*p3-c2*p2-c3*p1-p4)//4
                p5_first=(-c1*p4-c2*p3-c3*p2-c4*p1)%5
                for p5 in range(p_up[4]-(p_up[4]-p5_first)%5,p_low[4]-1,-5):
                    c5=(-c1*p4-c2*p3-c3*p2-c4*p1-p5)//5
                    p6_first=(-c1*p5-c2*p4-c3*p3-c4*p2-c5*p1)%6
                    for p6 in range(p_up[5]-(p_up[5]-p6_first)%6,p_low[5]-1,-6):
                        c6=(-c1*p5-c2*p4-c3*p3-c4*p2-c5*p1-p6)//6
                        coeff=[1,c1,c2,c3,c4,c5,c6,-c5,c4,-c3,c2,-c1,1] #higher power first
                        roots=sorted(abs(np.roots(coeff)))
                        l=roots[-1]
                        l2=roots[-2]
                        if l<lmin+0.00001 and np.polyval(coeff,lmin)>-0.00001 and l2<l:
                            coeff.extend([l,p1,p2,p3,p4,p5,p6])
                            working.append(coeff)

100%|██████████| 13/13 [00:34<00:00,  2.67s/it]


In [11]:
polyP=pd.DataFrame(working,columns=['c0','c1','c2','c3','c4','c5','c6','c7','c8','c9','c10','c11','c12','max root','p1','p2','p3','p4','p5','p6'])
print(polyP)

    c0  c1  c2  c3  c4  c5  c6  c7  c8  c9  c10  c11  c12  max root  p1  p2  \
0    1  -1  -2   2   2  -2  -2   2   2  -2   -2    1    1  1.229574   1   5   
1    1  -1   0   0  -1   2   0  -2  -1   0    0    1    1  1.240219   1   1   
2    1  -1   1   0  -1   0  -2   0  -1   0    1    1    1  1.249852   1  -1   
3    1   0  -4  -1   7   3  -8  -3   7   1   -4    0    1  1.252073   0   8   
4    1   0  -4   1   7  -3  -8   3   7  -1   -4    0    1  1.252073   0   8   
5    1   0  -3  -1   5   2  -6  -2   5   1   -3    0    1  1.252073   0   6   
6    1   0  -3   0   3   0  -2   0   3   0   -3    0    1  1.000069   0   6   
7    1   0  -3   0   4   0  -4   0   4   0   -3    0    1  1.000000   0   6   
8    1   0  -3   0   5   0  -6   0   5   0   -3    0    1  1.000000   0   6   
9    1   0  -3   1   5  -2  -6   2   5  -1   -3    0    1  1.252073   0   6   
10   1   0  -2  -2   3   2  -3  -2   3   2   -2    0    1  1.252073   0   4   
11   1   0  -2  -1   1   2   0  -2   1   1   -2    0

Some polynomials can be eliminated since they cleared the tests because of the margins we introduced.

In fact the polynomials 3,4,5,9,10,13,23,24,27,34,37,41,43,52 gives the candidate dilatation.
The polynomials 6,7,8,14,15,20,30,33,36,48,50 are cyclotomic and the polynomials 12,22,28,29,31,32,35,40,42,45,47,53,54 are non-primitiv.

We hence get the following list:

In [12]:
polyP=polyP.drop([3,4,5,6,7,8,9,10,12,13,14,15,20,22,23,24,27,28,29,30,31,32,33,34,35,36,37,40,41,42,43,45,47,48,50,52,53,54])
print(polyP)

    c0  c1  c2  c3  c4  c5  c6  c7  c8  c9  c10  c11  c12  max root  p1  p2  \
0    1  -1  -2   2   2  -2  -2   2   2  -2   -2    1    1  1.229574   1   5   
1    1  -1   0   0  -1   2   0  -2  -1   0    0    1    1  1.240219   1   1   
2    1  -1   1   0  -1   0  -2   0  -1   0    1    1    1  1.249852   1  -1   
11   1   0  -2  -1   1   2   0  -2   1   1   -2    0    1  1.236506   0   4   
16   1   0  -2   0   1  -1   0   1   1   0   -2    0    1  1.209829   0   4   
17   1   0  -2   0   1   1   0  -1   1   0   -2    0    1  1.209829   0   4   
18   1   0  -2   0   2  -1  -2   1   2   0   -2    0    1  1.159731   0   4   
19   1   0  -2   0   2   1  -2  -1   2   0   -2    0    1  1.159731   0   4   
21   1   0  -2   1   1  -2   0   2   1  -1   -2    0    1  1.236506   0   4   
25   1   0  -1  -1  -1   1   2  -1  -1   1   -1    0    1  1.243051   0   2   
26   1   0  -1  -1   0   1   0  -1   0   1   -1    0    1  1.204425   0   2   
38   1   0  -1   1  -1  -1   2   1  -1  -1   -1    0

In [13]:
maxpowersum=24

zeros=np.zeros(len(polyP.index),dtype=np.int64)
for i in range(7,maxpowersum):
    polyP.insert(len(polyP.columns),f'p{i}',zeros)

for idq in polyP.index:
    c=[polyP[f'c{i}'][idq] for i in range(0,13,1)]
    C=companion(c)
    B=companion(c)
    polyP.at[idq,'p1']=np.matrix.trace(C)
    for i in range(2,maxpowersum):
        B=np.matmul(B,C)
        polyP.at[idq,f'p{i}']=np.matrix.trace(B)

We now eliminate all combinations PQ for which an odd power sum is negative.

In [14]:
working=[]
for idp in polyP.index:
    for idq in polyQ.index:
        if  polyP["p1"][idp]+polyQ["p1"][idq]>=0 and polyP["p3"][idp]+polyQ["p3"][idq]>=0\
        and polyP["p5"][idp]+polyQ["p5"][idq]>=0 and polyP["p7"][idp]+polyQ["p7"][idq]>=0 and polyP["p9"][idp]+polyQ["p9"][idq]>=0\
        and polyP["p11"][idp]+polyQ["p11"][idq]>=0 and polyP["p13"][idp]+polyQ["p13"][idq]>=0 and polyP["p15"][idp]+polyQ["p15"][idq]>=0\
        and polyP["p17"][idp]+polyQ["p17"][idq]>=0 and polyP["p19"][idp]+polyQ["p19"][idq]>=0 and polyP["p21"][idp]+polyQ["p21"][idq]>=0\
        and polyP["p23"][idp]+polyQ["p23"][idq]>=0:
            working.append([idp,idq])

In [15]:
print(len(working))

38


We now apply Lemma 16 to count singularities.

In [16]:
working2=[]          # Create an empty list to push the polynomials passing the tests
maxpowersum=24
for row in working:  # We count singularities for every polynomial left
    singularities=0  
    order=[]         # List for the number of points with order exactly 2i+1, with i the index in the list
    for p in range(1,maxpowersum,2):  # We look at every odd power of f up to maxpowersum
        d_list=divisors(p)            # We take all divisors of p
        points=polyP[f'p{p}'][row[0]]+polyQ[f'p{p}'][row[1]]  # We take the p-th power sum, i.e. all the points fixed by p
        for d in range(len(d_list)-1):
            points-=order[(d_list[d]-1)//2]  #For every divisor of p, we substract the points fixed by f^p of order smaller than p
        order.append(points)   
        if (points/(p))%2==1:  #If there is an odd nbr of orbits of order p, by section 3.3.1 there are at least p singularities of order p.
            singularities+=p
    if singularities<=8:  #If there are less than 12 singularities, we push the polynomial in the working list
        working2.append(row)
working=working2  #We update the working list after the last test.

In [17]:
print(len(working))

0
